In [2]:
%pip install pandas
%pip install numpy
%pip install oracledb

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [3]:
# import modules
from datetime import datetime, timedelta
import pandas as pd
import numpy as np
import oracledb
import typing
import re
import random

In [4]:
# DB Stuff
ORACLE_HOST = "10.19.49.10"
ORACLE_PORT = 1522
ORACLE_SERVICE = "XE"
ORACLE_USER = "proiect"
ORACLE_PASSWORD = "proiect"

def db_get_connection() -> oracledb.Connection:
   """Se conecteaza la baza de date si returneaza conexiunea

   Returns:
      oracledb.Connection: Conexiunea la baza de date Oracle
   """
   dsn = f"{ORACLE_HOST}:{ORACLE_PORT}/{ORACLE_SERVICE}"
   print(f"Connecting to Oracle DB with DSN: '{dsn}'")
   connection: oracledb.Connection = oracledb.connect(user=ORACLE_USER, password=ORACLE_PASSWORD, dsn=dsn)
   print(f"Connected to Oracle DB with DSN: '{dsn}'; user: '{ORACLE_USER}'")
   return connection

def db_run_sql(conn: oracledb.Connection, 
               sql: str, 
               params: typing.Optional[dict] = None, 
               fetch: bool = False, 
               commit: bool = False,
               silent: bool = False) -> typing.Optional[list]: 
   """Executa o interogare SQL pe baza de date

   Args:
      conn (oracledb.Connection): Conexiunea la baza de date
      sql (str): Interogarea SQL de executat
      params (dict, optional): Parametrii pentru interogare. Defaults to None.
      fetch (bool, optional): Daca True, returneaza rezultatele interogarii. Defaults to False.
      commit (bool, optional): Daca True, face commit dupa executarea interogarii. Defaults to False.
      silent (bool, optional): Daca True, nu afiseaza interogarea SQL care se executa. Defaults to False.
      
   Returns:
      list: Rezultatele interogarii daca fetch este True, altfel None
   """
   if not silent:
      sql_print = str(sql).replace('\n', ' ').replace('\t', ' ')
      if len(sql_print) > 100:
         sql_print = sql_print[:100] + '...'
      print(f"Running SQL query: fetch={fetch}, commit={commit}, SQL: \"{sql_print}\", params: {params}")
   cursor = conn.cursor()
   if params:
      cursor.execute(sql, params)
   else:
      cursor.execute(sql)

   if fetch:
      return cursor.fetchall()

   if commit:
      conn.commit()
   return None

oracle_conn: oracledb.Connection = db_get_connection()

Connecting to Oracle DB with DSN: '10.19.49.10:1522/XE'
Connected to Oracle DB with DSN: '10.19.49.10:1522/XE'; user: 'proiect'


### Simulare tranzactii numerar pentru portofolii
!! Dupa rularea cell-ului de mai jos, executa statement-urile SQL generate

In [5]:
# generare 10 tranzactii pentru tabelul tranzactie_numerar
id_portofoliu_list: list[int] = [p[0] for p in db_run_sql(conn=oracle_conn, sql="SELECT id_portofoliu FROM portofoliu", fetch=True)] # type: ignore
cod_moneda_list: list[str] = [m[0] for m in db_run_sql(conn=oracle_conn, sql="SELECT cod_moneda FROM moneda", fetch=True)] # type: ignore

tipuri_miscare: list[str] = ['DEPOZIT', 'RETRAGERE', 'COMISION', 'DIVIDEND']
tranzactie_numerar_sql_inserts: list[str] = []
data_start = datetime.strptime('2026-01-01', '%Y-%m-%d').date()

for _ in range(10):
   id_portofoliu = random.choice(id_portofoliu_list)
   cod_moneda = random.choice(cod_moneda_list)
   tip_miscare = random.choices(tipuri_miscare, weights=[4, 2, 2, 2], k=1)[0]

   if tip_miscare == 'DEPOZIT':
      suma = round(random.uniform(3000, 25000), 2)
   elif tip_miscare == 'RETRAGERE':
      suma = round(random.uniform(200, 7000), 2)
   elif tip_miscare == 'COMISION':
      suma = round(random.uniform(5, 150), 2)
   else:
      suma = round(random.uniform(80, 3000), 2)

   data_tranzactie = data_start + timedelta(days=random.randint(0, 140))

   sql_insert = f"""
      INSERT INTO tranzactie_numerar (id_portofoliu, cod_moneda, tip_miscare, suma, data_tranzactie)
      VALUES ({id_portofoliu}, '{cod_moneda}', '{tip_miscare}', {suma}, TO_DATE('{data_tranzactie}', 'YYYY-MM-DD'));
   """.strip()
   sql_insert = re.sub(r"\s+", " ", sql_insert)
   tranzactie_numerar_sql_inserts.append(sql_insert)

print("\nGenerated SQL INSERT statements for 'tranzactie_numerar':")
for sql in tranzactie_numerar_sql_inserts:
   print(sql)

Running SQL query: fetch=True, commit=False, SQL: "SELECT id_portofoliu FROM portofoliu", params: None
Running SQL query: fetch=True, commit=False, SQL: "SELECT cod_moneda FROM moneda", params: None

Generated SQL INSERT statements for 'tranzactie_numerar':
INSERT INTO tranzactie_numerar (id_portofoliu, cod_moneda, tip_miscare, suma, data_tranzactie) VALUES (3, 'CHF', 'DEPOZIT', 20144.61, TO_DATE('2026-02-17', 'YYYY-MM-DD'));
INSERT INTO tranzactie_numerar (id_portofoliu, cod_moneda, tip_miscare, suma, data_tranzactie) VALUES (1, 'TWD', 'COMISION', 126.64, TO_DATE('2026-02-25', 'YYYY-MM-DD'));
INSERT INTO tranzactie_numerar (id_portofoliu, cod_moneda, tip_miscare, suma, data_tranzactie) VALUES (2, 'CHF', 'DIVIDEND', 890.47, TO_DATE('2026-02-23', 'YYYY-MM-DD'));
INSERT INTO tranzactie_numerar (id_portofoliu, cod_moneda, tip_miscare, suma, data_tranzactie) VALUES (5, 'USD', 'DEPOZIT', 22150.53, TO_DATE('2026-03-12', 'YYYY-MM-DD'));
INSERT INTO tranzactie_numerar (id_portofoliu, cod_moned